In [12]:
import numpy as np
import pandas as pd   # Imports all the libraries needed from numpy, pandas, and scikit

from sklearn.datasets import load_breast_cancer  # breast cancer data
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

In [13]:
class BreastCancerDataLoader:
# Class function for the breast cancer data
    def __init__(self, test_size=0.20, val_size=0.20, random_state=42):
        self.test_size = test_size
        self.val_size = val_size
        self.random_state = random_state

    def load_and_split(self):
# load dataset from breast cancer
        data = load_breast_cancer()
        X = data.data
        y = data.target
# trains x and y along with doing a validation test 
        X_trainval, X_test, y_trainval, y_test = train_test_split(
            X, y,
            test_size=self.test_size,
            random_state=self.random_state,
            stratify=y  # balances the classes proportions 
        )
# 2nd function to further train x and y plus the validations
        X_train, X_val, y_train, y_val = train_test_split(
            X_trainval,
            y_trainval,
            test_size=self.val_size,
            random_state=self.random_state,
            stratify=y_trainval
        )
# return all trained values
        return X_train, X_val, X_test, y_train, y_val, y_test

In [14]:
# this line of code builds the 3 machine learning models 
class ModelFactory:

    def __init__(self, random_state=42):
        self.random_state = random_state

    def build_models(self):

        logistic = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=8000))
        ])
# the svm and rbf kernel, it captures non linear decision boundaries
        svm = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(kernel="rbf", probability=True))
        ])
# the rf uses decision trees to improve the predictive performance of the 
        rf = RandomForestClassifier(
            n_estimators=600,
            random_state=self.random_state,
            class_weight="balanced"
        )
# store models 
        models = {
            "Logistic Regression": logistic,
            "SVM": svm,
            "Random Forest": rf
        }

        grids = {
# clf_c controls the regularizationn strength
            "Logistic Regression": {
                "clf__C": [0.01, 0.1, 1, 10, 100]
            },

            "SVM": {
                "clf__C": [0.1, 1, 10, 100, 1000],
                "clf__gamma": ["scale", 0.001, 0.01, 0.1]
            },
# random forest parameters help define the tree structure
            "Random Forest": {
                "max_depth": [None, 5, 10],
                "min_samples_split": [2, 5, 10],
                "min_samples_leaf": [1, 2, 4]
            }

        }
# return the models and grids labeled earlier
        return models, grids

In [15]:
class ModelEvaluator:

    def evaluate(self, model, X_train, y_train, X_test, y_test):
# train the model based on the training datset
        model.fit(X_train, y_train)

        preds = model.predict(X_test)
# calculates the common classification metrics 
        accuracy = accuracy_score(y_test, preds)
        precision = precision_score(y_test, preds)
        recall = recall_score(y_test, preds)
        f1 = f1_score(y_test, preds)

        if hasattr(model, "predict_proba"):
            probs = model.predict_proba(X_test)[:,1]
            roc = roc_auc_score(y_test, probs)
        else:
            roc = None

        conf = confusion_matrix(y_test, preds)
# print all the results
        print("Accuracy:", accuracy)
        print("Precision:", precision)
        print("Recall:", recall)
        print("F1:", f1)
        print("ROC AUC:", roc)
        print("Confusion Matrix")
        print(conf)

        print("Classification Report")
        print(classification_report(y_test, preds))

        return accuracy, precision, recall, f1, roc

In [16]:
# create variable to load the data
loader = BreastCancerDataLoader()
# load and split the data
X_train, X_val, X_test, y_train, y_val, y_test = loader.load_and_split()

In [17]:
#variable for the model factory object
factory = ModelFactory()
# grids for the hyperperameters/ builds models
models, grids = factory.build_models()

In [18]:
tuned_models = {}

for name in models:

    print("Tuning:", name)
# tests different hyperperameters combos to generate the best performing model
    grid = GridSearchCV(
        models[name],
        grids[name],
        cv=5,
        scoring="roc_auc",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    tuned_models[name] = grid.best_estimator_

    print("Best Params:", grid.best_params_)
    print("Best CV ROC AUC:", grid.best_score_)

Tuning: Logistic Regression
Best Params: {'clf__C': 0.1}
Best CV ROC AUC: 0.9940380338931064
Tuning: SVM
Best Params: {'clf__C': 1, 'clf__gamma': 'scale'}
Best CV ROC AUC: 0.9943777317690362
Tuning: Random Forest
Best Params: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best CV ROC AUC: 0.9946169772256729


In [19]:
evaluator = ModelEvaluator()

results = {}

for name in tuned_models:

   
    print(name)
    

    acc, prec, rec, f1, roc = evaluator.evaluate(
        tuned_models[name],
        X_train,
        y_train,
        X_val,
        y_val
    )

    results[name] = roc

Logistic Regression
Accuracy: 0.978021978021978
Precision: 0.9661016949152542
Recall: 1.0
F1: 0.9827586206896551
ROC AUC: 0.9969040247678018
Confusion Matrix
[[32  2]
 [ 0 57]]
Classification Report
              precision    recall  f1-score   support

           0       1.00      0.94      0.97        34
           1       0.97      1.00      0.98        57

    accuracy                           0.98        91
   macro avg       0.98      0.97      0.98        91
weighted avg       0.98      0.98      0.98        91

SVM
Accuracy: 0.978021978021978
Precision: 0.9824561403508771
Recall: 0.9824561403508771
F1: 0.9824561403508771
ROC AUC: 0.9969040247678018
Confusion Matrix
[[33  1]
 [ 1 56]]
Classification Report
              precision    recall  f1-score   support

           0       0.97      0.97      0.97        34
           1       0.98      0.98      0.98        57

    accuracy                           0.98        91
   macro avg       0.98      0.98      0.98        91
weig

In [20]:
# prints the model with the highest ROC/AUC score aka the best model
best_model_name = max(results, key=results.get)

print("Best Model:", best_model_name)

Best Model: Logistic Regression


In [21]:
# combines all the training and validation models to retain the best model  
best_model = tuned_models[best_model_name]

evaluator.evaluate(
    best_model,
    np.vstack((X_train, X_val)),
    np.concatenate((y_train, y_val)),
    X_test,
    y_test
)

Accuracy: 0.9736842105263158
Precision: 0.9726027397260274
Recall: 0.9861111111111112
F1: 0.9793103448275862
ROC AUC: 0.9957010582010581
Confusion Matrix
[[40  2]
 [ 1 71]]
Classification Report
              precision    recall  f1-score   support

           0       0.98      0.95      0.96        42
           1       0.97      0.99      0.98        72

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114



(0.9736842105263158,
 0.9726027397260274,
 0.9861111111111112,
 0.9793103448275862,
 np.float64(0.9957010582010581))